# 194. Agent 工具副作用：Saga、补偿与幂等怎样实现？

> **面试问题：当 Agent 连续调用支付、库存、物流等有副作用工具时，怎样防重复执行、处理半完成，并把失败安全地收束？**

## 先给结论

高质量回答需要同时说明目标状态、可执行策略、状态版本、失败回滚和可复放评测。下面不用 Agent 框架或远程工具，而是用受控的内存模型把核心合同写出来；断言只证明教学实现的不变量，不能替代真实服务的隔离、审计、权限与压测。

## 一手资料

- [ReAct](https://arxiv.org/abs/2210.03629)
- [Toolformer](https://arxiv.org/abs/2302.04761)
- [AgentDojo](https://arxiv.org/abs/2406.13352)

In [ ]:
notebook_contract = {"mode": "in-memory-demo", "oracle": "explicit-assertions", "production": "needs-isolation"}  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["mode"] == "in-memory-demo"  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["oracle"] == "explicit-assertions"  # 执行本行的状态、计算或校验逻辑。
assert "isolation" in notebook_contract["production"]  # 执行本行的状态、计算或校验逻辑。
assert len(notebook_contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 问题拆解：Agent 的工具副作用如何避免半完成

预订、扣款、发货等跨工具流程可能在中间失败。Saga 的工程思想是：每个已完成动作准备一个语义明确的补偿动作，并按相反顺序执行；补偿不是数据库事务，也不能保证外部世界完全可逆，所以还必须有幂等键和人工对账。


In [ ]:
from dataclasses import dataclass  # 执行本行的状态、计算或校验逻辑。
@dataclass(frozen=True)  # 执行本行的状态、计算或校验逻辑。
class SagaStep:  # 执行本行的状态、计算或校验逻辑。
    name: str  # 执行本行的状态、计算或校验逻辑。
    key: str  # 执行本行的状态、计算或校验逻辑。
    amount: int  # 执行本行的状态、计算或校验逻辑。
@dataclass  # 执行本行的状态、计算或校验逻辑。
class Ledger:  # 执行本行的状态、计算或校验逻辑。
    balance: int  # 执行本行的状态、计算或校验逻辑。
    reserved: int = 0  # 执行本行的状态、计算或校验逻辑。
    shipped: bool = False  # 执行本行的状态、计算或校验逻辑。
    seen: set = None  # 执行本行的状态、计算或校验逻辑。
    audit: list = None  # 执行本行的状态、计算或校验逻辑。
    def __post_init__(self):  # 执行本行的状态、计算或校验逻辑。
        self.seen = set() if self.seen is None else self.seen  # 执行本行的状态、计算或校验逻辑。
        self.audit = [] if self.audit is None else self.audit  # 执行本行的状态、计算或校验逻辑。
ledger = Ledger(balance=100)  # 执行本行的状态、计算或校验逻辑。
assert ledger.balance == 100  # 执行本行的状态、计算或校验逻辑。
assert ledger.reserved == 0  # 执行本行的状态、计算或校验逻辑。
assert ledger.shipped is False  # 执行本行的状态、计算或校验逻辑。


## 2. 幂等写入：重试不能重复扣款

所有可能被网络重试的副作用都需要业务幂等键。这里用 `seen` 模拟去重表；真实服务需要在强一致的存储中记录 request id、结果与过期策略，不能只依赖进程内内存。


In [ ]:
def reserve(ledger, key, amount):  # 执行本行的状态、计算或校验逻辑。
    if key in ledger.seen:  # 执行本行的状态、计算或校验逻辑。
        return "deduplicated"  # 执行本行的状态、计算或校验逻辑。
    if amount <= 0 or ledger.balance < amount:  # 执行本行的状态、计算或校验逻辑。
        raise ValueError("余额或金额不合法")  # 执行本行的状态、计算或校验逻辑。
    ledger.seen.add(key)  # 执行本行的状态、计算或校验逻辑。
    ledger.balance -= amount  # 执行本行的状态、计算或校验逻辑。
    ledger.reserved += amount  # 执行本行的状态、计算或校验逻辑。
    ledger.audit.append(("reserve", key, amount))  # 执行本行的状态、计算或校验逻辑。
    return "reserved"  # 执行本行的状态、计算或校验逻辑。
assert reserve(ledger, "reserve-1", 30) == "reserved"  # 执行本行的状态、计算或校验逻辑。
assert reserve(ledger, "reserve-1", 30) == "deduplicated"  # 执行本行的状态、计算或校验逻辑。
assert (ledger.balance, ledger.reserved) == (70, 30)  # 执行本行的状态、计算或校验逻辑。


## 3. 补偿动作：只撤销已成功且可撤销的步骤

补偿应带自己的幂等键，并记录回滚原因。它不能简单“把状态设回初始值”，否则会覆盖并发写；这里仅退还本 saga 已保留金额。真正的支付/物流系统还要接入对账与人工例外队列。


In [ ]:
def compensate_reserve(ledger, key, amount):  # 执行本行的状态、计算或校验逻辑。
    if key in ledger.seen:  # 执行本行的状态、计算或校验逻辑。
        return "deduplicated"  # 执行本行的状态、计算或校验逻辑。
    if ledger.reserved < amount:  # 执行本行的状态、计算或校验逻辑。
        raise ValueError("没有足够的可补偿预留")  # 执行本行的状态、计算或校验逻辑。
    ledger.seen.add(key)  # 执行本行的状态、计算或校验逻辑。
    ledger.reserved -= amount  # 执行本行的状态、计算或校验逻辑。
    ledger.balance += amount  # 执行本行的状态、计算或校验逻辑。
    ledger.audit.append(("compensate_reserve", key, amount))  # 执行本行的状态、计算或校验逻辑。
    return "compensated"  # 执行本行的状态、计算或校验逻辑。
assert compensate_reserve(ledger, "undo-reserve-1", 30) == "compensated"  # 执行本行的状态、计算或校验逻辑。
assert (ledger.balance, ledger.reserved) == (100, 0)  # 执行本行的状态、计算或校验逻辑。
assert compensate_reserve(ledger, "undo-reserve-1", 30) == "deduplicated"  # 执行本行的状态、计算或校验逻辑。


## 4. 第二步工具：失败须显式返回

Agent 不应把工具输出当成可信自然语言。每个工具应返回结构化成功/失败，失败后 loop 决定能否重试、补偿还是升级人工。本例把物流开关固定为布尔值，专门演示失败路径。


In [ ]:
def ship(ledger, key, available):  # 执行本行的状态、计算或校验逻辑。
    if key in ledger.seen:  # 执行本行的状态、计算或校验逻辑。
        return {"ok": ledger.shipped, "reason": "deduplicated"}  # 执行本行的状态、计算或校验逻辑。
    ledger.seen.add(key)  # 执行本行的状态、计算或校验逻辑。
    if not available:  # 执行本行的状态、计算或校验逻辑。
        ledger.audit.append(("ship", key, "unavailable"))  # 执行本行的状态、计算或校验逻辑。
        return {"ok": False, "reason": "carrier_unavailable"}  # 执行本行的状态、计算或校验逻辑。
    ledger.shipped = True  # 执行本行的状态、计算或校验逻辑。
    ledger.audit.append(("ship", key, "shipped"))  # 执行本行的状态、计算或校验逻辑。
    return {"ok": True, "reason": "shipped"}  # 执行本行的状态、计算或校验逻辑。
failed_shipping = ship(ledger, "ship-1", False)  # 执行本行的状态、计算或校验逻辑。
assert failed_shipping["ok"] is False  # 执行本行的状态、计算或校验逻辑。
assert ledger.shipped is False  # 执行本行的状态、计算或校验逻辑。
assert failed_shipping["reason"] == "carrier_unavailable"  # 执行本行的状态、计算或校验逻辑。


## 5. Saga loop：失败时按逆序补偿

编排器必须只补偿已确认成功的前置步骤，并把失败原因连同每个补偿结果写入 trace。对不可逆步骤，不应偷偷伪造成功，而要进入人工处理或后续 reconcile 流程。


In [ ]:
def book_with_saga(ledger, amount, shipping_available):  # 执行本行的状态、计算或校验逻辑。
    reserve_key = "reserve-2"  # 执行本行的状态、计算或校验逻辑。
    reserve(ledger, reserve_key, amount)  # 执行本行的状态、计算或校验逻辑。
    shipping = ship(ledger, "ship-2", shipping_available)  # 执行本行的状态、计算或校验逻辑。
    if not shipping["ok"]:  # 执行本行的状态、计算或校验逻辑。
        compensate_reserve(ledger, "undo-reserve-2", amount)  # 执行本行的状态、计算或校验逻辑。
        return {"ok": False, "reason": shipping["reason"], "compensated": True}  # 执行本行的状态、计算或校验逻辑。
    return {"ok": True, "reason": "completed", "compensated": False}  # 执行本行的状态、计算或校验逻辑。
outcome = book_with_saga(ledger, 20, False)  # 执行本行的状态、计算或校验逻辑。
assert outcome["ok"] is False  # 执行本行的状态、计算或校验逻辑。
assert outcome["compensated"] is True  # 执行本行的状态、计算或校验逻辑。
assert (ledger.balance, ledger.reserved) == (100, 0)  # 执行本行的状态、计算或校验逻辑。


## 6. 重试与幂等：再次运行不会再次产生扣款

失败后的网络重试是常态。由于 reserve、ship 与 compensate 都带独立幂等键，重复运行同一 saga 不会重复扣款或重复退款；但生产系统仍要定义 key 的作用域和有效期。


In [ ]:
retry = book_with_saga(ledger, 20, False)  # 执行本行的状态、计算或校验逻辑。
assert retry["ok"] is False  # 执行本行的状态、计算或校验逻辑。
assert (ledger.balance, ledger.reserved) == (100, 0)  # 执行本行的状态、计算或校验逻辑。
assert ledger.audit.count(("compensate_reserve", "undo-reserve-2", 20)) == 1  # 执行本行的状态、计算或校验逻辑。


## 7. 成功路径：提交后不再执行补偿

成功 saga 也要验证终态：金额已保留、物流已确认、没有补偿事件。这里的余额与预留只是简化账本；真实业务通常以订单状态机、支付授权/捕获和异步事件最终一致地表达。


In [ ]:
success_ledger = Ledger(balance=50)  # 执行本行的状态、计算或校验逻辑。
success = book_with_saga(success_ledger, 20, True)  # 执行本行的状态、计算或校验逻辑。
assert success["ok"] is True  # 执行本行的状态、计算或校验逻辑。
assert success_ledger.shipped is True  # 执行本行的状态、计算或校验逻辑。
assert (success_ledger.balance, success_ledger.reserved) == (30, 20)  # 执行本行的状态、计算或校验逻辑。


## 8. 审计与安全：工具结果与策略必须独立验证

工具型 Agent 还需要把每次副作用同用户授权、策略版本和工具 schema 绑定，且把外部文本视为不可信输入。AgentDojo 这类评测提醒我们：任务完成、权限遵从和抗提示注入是不同目标。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
artifact = {"saga": "booking-v1", "policy": "payment-confirmed-v1", "events": success_ledger.audit, "terminal": success}  # 执行本行的状态、计算或校验逻辑。
audit_hash = hashlib.sha256(json.dumps(artifact, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert artifact["terminal"]["reason"] == "completed"  # 执行本行的状态、计算或校验逻辑。
assert len(audit_hash) == 64  # 执行本行的状态、计算或校验逻辑。
assert any(event[0] == "ship" for event in success_ledger.audit)  # 执行本行的状态、计算或校验逻辑。


## 面试收束

按“任务目标 → 显式状态 → 动作前策略门禁 → 成功 oracle → 失败和重试 → 指标与版本化制品”的顺序回答。不要把一次文本看起来合理的演示当成可靠性证明：要独立检查状态、权限、不可逆副作用与多次运行的一致性。
